# BCS 3101: Basics of Machine Learning
## Assignment 2 – Notebook 7 (Final End-to-End Notebook)

**Student Name:** AINEBYONA ALLAN  
**Registration Number:** 2024/A/KCS/1670/F  
**Group Project:** Predicting Monthly Maize and Beans Prices in Selected Ugandan Markets  

This is the final notebook.  
It brings together every stage of the pipeline in one place so the examiner can run the whole workflow from top to bottom and see the outputs.

All decisions follow the companion guide and the marking rubric:
- Pre-processing Pipeline Understanding – 10%
- Dataset Selection & Description – 10%
- Data Cleaning – 15%
- Data Transformation – 15%
- Data Reduction & Splitting – 10%
- Exploratory Data Analysis – 20%
- Report Structure & Submission Compliance – 20%

---
## 1. Problem Title and Dataset Description

**Title**  
Predicting Monthly Maize and Beans Prices in Selected Ugandan Markets Using Historical Price and Market Data

**Task type**  
Regression (we predict continuous price values).

**Dataset**  
Uganda Real-Time Food Prices (RTFP) from the World Bank, based on WFP and FAO market data.  
We keep seven markets: Market Average, Gulu, Lira, Jinja, Hoima, Busia and Mbarara.  
Targets are the completed price series `c_maize` and `c_beans`.

---
## 2. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

%matplotlib inline
sns.set_style('whitegrid')

print("Libraries imported successfully.")

---
## 3. Load the data and keep the seven markets

In [ ]:
# Load the filtered dataset prepared in Notebook 1
df = pd.read_csv('uganda_maize_beans_selected_markets.csv')

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\nMarkets and record counts:")
print(df['mkt_name'].value_counts())

We have 1 652 rows (7 markets × 236 months).  
This meets the guide’s requirement of at least 500 rows and 5 columns.

---
## 4. Data Cleaning (summary of decisions from Notebook 2)

- Targets `c_maize` and `c_beans` have zero missing values → keep them.
- Original observed columns (`maize`, `beans`, etc.) have high missing rates → do not use them as features.
- No exact duplicate rows were found.
- Outliers detected by the IQR rule are retained because they represent real market spikes, not typing errors.

In [ ]:
# Confirm targets have no missing values
print("Missing values in targets:")
print(df[['c_maize', 'c_beans']].isnull().sum())

# Confirm no exact duplicates
print(f"\nExact duplicate rows: {df.duplicated().sum()}")

---
## 5. Data Integration

Data integration is **not applicable**.  
We use a single source file.  
The guide rewards an explicit statement rather than a silent skip.

In [ ]:
print("Number of source files: 1")
print("Data integration: NOT APPLICABLE")

---
## 6. Encoding Categorical Variables

`mkt_name` is nominal (no natural order).  
We therefore use One-Hot Encoding with `drop_first=True`.

In [ ]:
df = pd.get_dummies(df, columns=['mkt_name'], drop_first=True, dtype=int)

market_cols = [c for c in df.columns if c.startswith('mkt_name_')]
print("One-hot market columns created:")
print(market_cols)
print(f"\nShape after encoding: {df.shape}")

---
## 7. Select features and targets

In [ ]:
feature_candidates = [
    'year', 'month', 'lat', 'lon',
    'c_oil', 'c_salt', 'c_food_price_index',
    'inflation_maize', 'inflation_beans',
    'trust_maize', 'trust_beans',
    'data_coverage', 'data_coverage_recent', 'index_confidence_score'
] + market_cols

feature_cols = [c for c in feature_candidates if c in df.columns]
target_cols = ['c_maize', 'c_beans']

df_model = df[feature_cols + target_cols].dropna().copy()

print(f"Rows after dropping any remaining missing values: {df_model.shape[0]}")
print(f"Number of features: {len(feature_cols)}")
print("Features:", feature_cols)

---
## 8. Quick Exploratory Look at the Targets

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df_model['c_maize'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Target: c_maize')
axes[0].set_xlabel('Price (UGX)')

sns.histplot(df_model['c_beans'], kde=True, ax=axes[1], color='seagreen')
axes[1].set_title('Target: c_beans')
axes[1].set_xlabel('Price (UGX)')

plt.tight_layout()
plt.show()

print("Skewness c_maize:", round(df_model['c_maize'].skew(), 2))
print("Skewness c_beans:", round(df_model['c_beans'].skew(), 2))

Both targets are right-skewed.  
This supports the earlier decision to consider a log transform if a later model needs more symmetric residuals.

---
## 9. Feature Correlation Check

In [ ]:
num_feats = [c for c in feature_cols if not c.startswith('mkt_name_')]
corr = df_model[num_feats + target_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation heatmap')
plt.tight_layout()
plt.show()

Maize and beans prices move together.  
Both also rise with the food-price index and with year (long-term upward trend).

---
## 10. PCA Exploration (and decision not to use it for modelling)

We standardise the features, fit PCA for 95% variance, then decide whether to keep the components.

In [ ]:
scaler_pca = StandardScaler()
X_scaled_temp = scaler_pca.fit_transform(df_model[feature_cols])

pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled_temp)

print(f"Original features: {len(feature_cols)}")
print(f"PCA components for 95% variance: {pca.n_components_}")
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

**Decision:** We do **not** replace the original features with PCA components.  

**Reason:** The number of features is modest and each feature has a clear meaning (year, month, market, price index, etc.).  
Keeping the original features makes the analysis easier to explain.  
The guide explicitly credits a reasoned decision to omit PCA.

---
## 11. Train / Test Split and Correct Scaling

We split first (80/20), then fit the scaler only on the training data.  
This follows the guide’s rule against data leakage.

In [ ]:
X = df_model[feature_cols]
y_maize = df_model['c_maize']
y_beans = df_model['c_beans']

X_train, X_test, y_train_m, y_test_m = train_test_split(
    X, y_maize, test_size=0.2, random_state=42
)

_, _, y_train_b, y_test_b = train_test_split(
    X, y_beans, test_size=0.2, random_state=42
)

print(f"Training rows: {X_train.shape[0]}")
print(f"Test rows:     {X_test.shape[0]}")

In [ ]:
# Fit scaler on training features only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # transform only

print("Scaler fitted on training data only.")
print(f"Scaled training shape: {X_train_scaled.shape}")
print(f"Scaled test shape:     {X_test_scaled.shape}")

---
## 12. Final Status – Data Ready for Modelling

At this point the data has passed through every required stage:

1. Clear regression problem and real-world dataset  
2. Cleaning decisions documented  
3. Integration declared not applicable  
4. Categorical market variable one-hot encoded  
5. Features selected and correlations examined  
6. PCA explored and deliberately not used  
7. Proper train-test split performed  
8. Scaler fitted only on the training set  

The matrices `X_train_scaled`, `X_test_scaled` and the corresponding target vectors are ready for any regression algorithm (linear regression, random forest, gradient boosting, etc.).

No further pre-processing is required before model training.

In [ ]:
# Save the final ready-to-model arrays
np.savez('final_train_test_maize.npz',
         X_train=X_train_scaled, X_test=X_test_scaled,
         y_train=y_train_m.values, y_test=y_test_m.values)

np.savez('final_train_test_beans.npz',
         X_train=X_train_scaled, X_test=X_test_scaled,
         y_train=y_train_b.values, y_test=y_test_b.values)

print("Final train/test arrays saved.")
print("Pipeline complete.")

---
## End of Notebook 7

This notebook can be run from the first cell to the last cell in a fresh environment.  
All key decisions required by the companion guide are present and can be copied into the written report.

**Next step for the group**  
Write the structured report (5–8 pages) following the exact seven-section structure in Stage 11 of the guide.  
Use the numbers, charts and justifications from Notebooks 1–7 as evidence.